<a href="https://colab.research.google.com/github/PRAJEENS2024/prajeen-codeboosters-2026/blob/main/Day%207/Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers -q
print("Installation complete")
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print("All libraries imported successfully")
print(f"ChromaDB version:{chromadb.__version__}")
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(name="my_collection")



Installation complete
All libraries imported successfully
ChromaDB version:1.5.9


In [ ]:
notes_df = pd.read_csv("college_notes.csv")
print(f"\nDataset loaded. Shape: {notes_df.shape}, Columns: {list(notes_df.columns)}")



Dataset loaded. Shape: (10, 4), Columns: ['note_id', 'subject', 'topic', 'content']


In [ ]:
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [{
    "subject": row['subject'],
    "topic": row['topic']
} for _, row in notes_df.iterrows()]
print(f"Documents prepared: {len(all_documents)}")
print(f"IDs prepared: {len(all_ids)}")
print(f"Metadata prepared: {len(all_metadatas)}")


Documents prepared: 10
IDs prepared: 10
Metadata prepared: 10


In [ ]:
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")



Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Produces vectors of size: 384 dimensions


/tmp/ipykernel_15189/3457278829.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


In [ ]:
print("\nInitializing ChromaDB client with Cosine Similarity...")
chroma_client = chromadb.Client()
collection_name = "college_notes_cosine"
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass
collection = chroma_client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
print(f"Collection '{collection_name}' created with cosine distance.")


Initializing ChromaDB client with Cosine Similarity...
Collection 'college_notes_cosine' created with cosine distance.


In [ ]:
print("\nAdding all notes to ChromaDB collection...")
collection.add(
    documents=all_documents,
    ids=all_ids,
    metadatas=all_metadatas
)
print(f"Documents added to Collection. Total: {collection.count()}")



Adding all notes to ChromaDB collection...
Documents added to Collection. Total: 10


In [ ]:
collection.add(documents=all_documents, ids=all_ids, metadatas=all_metadatas)
filtered_query_text = "Model evaluation metrics like accuracy for classification performance"
filtered_query_embedding = model.encode(filtered_query_text).tolist()
print(f"\n--- Performing Filtered Semantic Search (Cosine) for: '{filtered_query_text}' ---")
filtered_results = collection.query(query_embeddings=[filtered_query_embedding], n_results=2, where={"subject": "Machine Learning"}, include=['documents', 'distances', 'metadatas'])
for rank, (doc, doc_id, doc_dist, doc_meta) in enumerate(zip(filtered_results['documents'][0], filtered_results['ids'][0], filtered_results['distances'][0], filtered_results['metadatas'][0])):
    print(f"Rank: {rank} | ID: {doc_id} | Distance: {doc_dist:.4f}")
    print(f"Subject: {doc_meta['subject']} | Topic: {doc_meta['topic']}")
    print(f"Document: {doc[:150]}...")
    print('-' * 30)


--- Performing Filtered Semantic Search (Cosine) for: 'Model evaluation metrics like accuracy for classification performance' ---
Rank: 0 | ID: N007 | Distance: 0.2212
Subject: Machine Learning | Topic: Model Evaluation
Document: Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R...
------------------------------
Rank: 1 | ID: N009 | Distance: 0.6834
Subject: Machine Learning | Topic: Decision Trees
Document: A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at eac...
------------------------------
